# Task 2. 누적 적재 데이터 최신성·DB 추출·시계열 검증

- 이 노트북은 로컬 산출물이 실제로 어떤 Delta/DuckDB pair에서 읽혔는지 먼저 확인한 뒤, 선택된 pair 기준으로 raw Delta → DuckDB downstream 추출 → 최신성 → hourly 시계열 연속성을 점검합니다.

- 검증 범위는 local evidence입니다. 외부 RPC provider를 새로 호출하지 않으므로 live streaming 또는 production realtime 검증으로 해석하지 않습니다.

## 운영 규칙

- `NOTEBOOK_DELTA_LOGS_PATH`, `NOTEBOOK_DUCKDB_PATH`를 둘 다 지정하면 
    그 pair를 우선 사용합니다.

- 명시 경로가 없으면 현재 Python raw schema와 맞는 최신 local pair를 먼저 선택합니다. 
    현재 repo에서는 `data/delta/ethereum_logs_v2`,           
    `data/analytics/ethereum_analytics_v2.duckdb`가 이 조건을 만족할 수 있습니다.

- `data/delta/ethereum_logs`, `data/analytics/ethereum_analytics.duckdb`는 
    canonical default 이름이지만, 로컬에 구 schema 또는 오래된 smoke output이 남아 있으면
    stale/legacy로 표시합니다.

- DuckDB view가 `/opt/airflow/...` 같은 container 절대경로를 
    참조하면 다른 실행 환경에서 깨질 수 있으므로 query error를 숨기지 않습니다.

## 0. import 및 공통 helper

In [12]:
from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass
from datetime import UTC, datetime
from decimal import Decimal
from pathlib import Path
from typing import Any

import duckdb
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(value: object) -> None:
        print(value)

DISPLAY_TZ = "Asia/Seoul"
FRESHNESS_WARNING_HOURS = 24
SERIES_LOOKBACK_ROWS = 24
EXPECTED_RELATIONS = (
    "ethereum_logs",
    "erc20_transfers",
    "tether_treasury_flow",
    "tether_treasury_flow_quality_summary",
)


def find_project_root(start: Path | None = None) -> Path:
    """목적: 실행 위치와 무관하게 repository root를 찾는다.

    불변: `pyproject.toml`과 `src/cryptoquant_pipeline`가 함께 있는 경로만 root로 인정한다.
    실패: root를 못 찾으면 잘못된 data path를 추정하지 않고 즉시 중단한다.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "src" / "cryptoquant_pipeline"
        ).exists():
            return candidate
    raise RuntimeError("pyproject.toml과 src/cryptoquant_pipeline을 가진 repository root를 찾지 못했다.")


PROJECT_ROOT = find_project_root()
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cryptoquant_pipeline.delta_writer import (  # noqa: E402
    NATURAL_KEY_COLUMNS,
    PARTITION_COLUMNS,
    ethereum_logs_schema,
)


@dataclass(frozen=True)
class CandidatePair:
    """목적: raw Delta와 DuckDB analytics 파일을 하나의 검증 단위로 묶는다."""

    name: str
    delta_path: Path
    duckdb_path: Path
    source: str


def dedupe_paths(paths: list[Path]) -> list[Path]:
    seen: set[str] = set()
    result: list[Path] = []
    for path in paths:
        key = str(path)
        if key not in seen:
            seen.add(key)
            result.append(path)
    return result


def first_existing(candidates: list[Path]) -> Path | None:
    for candidate in dedupe_paths(candidates):
        if candidate.exists():
            return candidate.resolve()
    return None


def relative_candidates(relative_path: str) -> list[Path]:
    return dedupe_paths(
        [
            PROJECT_ROOT / relative_path,
            Path("/workspace") / relative_path,
            Path("/opt/airflow") / relative_path,
        ]
    )


def as_utc_datetime(value: Any) -> datetime | None:
    if value is None:
        return None
    if isinstance(value, datetime):
        if value.tzinfo is None:
            return value.replace(tzinfo=UTC)
        return value.astimezone(UTC)
    text = str(value)
    if not text or text.lower() == "nat":
        return None
    parsed = datetime.fromisoformat(text.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        return parsed.replace(tzinfo=UTC)
    return parsed.astimezone(UTC)


def hours_between(later: datetime | None, earlier: datetime | None) -> float | None:
    if later is None or earlier is None:
        return None
    return round((later - earlier).total_seconds() / 3600, 3)


def format_value(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, datetime):
        utc_value = as_utc_datetime(value)
        return "" if utc_value is None else utc_value.isoformat().replace("+00:00", "Z")
    if isinstance(value, float):
        return f"{value:.3f}"
    if isinstance(value, Decimal):
        return str(value)
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    text = str(value).replace("\n", " ").replace("|", "\\|")
    return text if len(text) <= 180 else f"{text[:177]}..."


def display_records(title: str, records: list[dict[str, Any]], *, limit: int = 50) -> None:
    display(Markdown(f"### {title}"))
    if not records:
        display(Markdown("_no rows_"))
        return
    shown = records[:limit]
    columns = list(shown[0])
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = ["| " + " | ".join(format_value(row.get(column)) for column in columns) + " |" for row in shown]
    if len(records) > limit:
        body.append("| ... | " + f"{len(records) - limit} more rows |" + " |" * max(0, len(columns) - 2))
    display(Markdown("\n".join([header, sep, *body])))


def to_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    """목적: DuckDB 조회 결과를 노트북에서 바로 검토 가능한 DataFrame으로 변환한다."""
    return pd.DataFrame.from_records(records)


def display_dataframe(title: str, records: list[dict[str, Any]], *, limit: int = 50) -> pd.DataFrame:
    """목적: DB 추출 결과와 시간대별 추이를 pandas DataFrame으로 표시한다.

    불변: 원본 records는 수정하지 않고, 화면 표시용 DataFrame만 생성한다.
    """
    display(Markdown(f"### {title}"))
    df = to_dataframe(records)
    display(df.head(limit))
    return df


def fetch_records(
    con: duckdb.DuckDBPyConnection,
    sql: str,
    params: list[Any] | None = None,
) -> tuple[list[dict[str, Any]], str | None]:
    try:
        cursor = con.execute(sql, params or [])
        columns = [column[0] for column in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()], None
    except Exception as exc:
        return [], f"{type(exc).__name__}: {exc}"


def fetch_scalar(
    con: duckdb.DuckDBPyConnection,
    sql: str,
    params: list[Any] | None = None,
) -> tuple[Any | None, str | None]:
    try:
        return con.execute(sql, params or []).fetchone()[0], None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def compare_schema_fields(actual_schema: Any) -> tuple[list[dict[str, Any]], bool]:
    expected_schema = ethereum_logs_schema(pa)
    actual_by_name = {field.name: field for field in actual_schema}
    expected_by_name = {field.name: field for field in expected_schema}
    all_columns = sorted(set(actual_by_name) | set(expected_by_name))
    records: list[dict[str, Any]] = []
    blocking_statuses = {"MISSING_IN_DATA", "TYPE_MISMATCH", "NULLABILITY_MISMATCH"}

    for column_name in all_columns:
        expected = expected_by_name.get(column_name)
        actual = actual_by_name.get(column_name)
        if expected is None:
            status = "EXTRA_IN_DATA"
        elif actual is None:
            status = "MISSING_IN_DATA"
        elif str(expected.type) != str(actual.type):
            status = "TYPE_MISMATCH"
        elif expected.nullable != actual.nullable:
            status = "NULLABILITY_MISMATCH"
        else:
            status = "PASS"
        records.append(
            {
                "column_name": column_name,
                "expected_type": str(expected.type) if expected else None,
                "actual_type": str(actual.type) if actual else None,
                "expected_nullable": expected.nullable if expected else None,
                "actual_nullable": actual.nullable if actual else None,
                "status": status,
            }
        )

    current = not any(record["status"] in blocking_statuses for record in records)
    return records, current


display(
    Markdown(
        "### Runtime context\n"
        "```text\n"
        f"project_root = {PROJECT_ROOT}\n"
        f"partition_columns = {PARTITION_COLUMNS}\n"
        f"natural_key = {NATURAL_KEY_COLUMNS}\n"
        "```"
    )
)

### Runtime context
```text
project_root = /workspace
partition_columns = ['block_date_utc']
natural_key = ('chain_id', 'transaction_hash', 'log_index')
```

## 1. 데이터셋 후보 인벤토리와 선택

In [13]:
def candidate_pairs() -> list[CandidatePair]:
    pairs: list[CandidatePair] = []
    env_delta = os.environ.get("NOTEBOOK_DELTA_LOGS_PATH")
    env_duckdb = os.environ.get("NOTEBOOK_DUCKDB_PATH")
    if env_delta and env_duckdb:
        delta_path = Path(env_delta).expanduser()
        duckdb_path = Path(env_duckdb).expanduser()
        if delta_path.exists() and duckdb_path.exists():
            pairs.append(CandidatePair("env_override", delta_path.resolve(), duckdb_path.resolve(), "env"))

    definitions = [
        (
            "latest_v2_local",
            "data/delta/ethereum_logs_v2",
            "data/analytics/ethereum_analytics_v2.duckdb",
            "local latest observed Airflow output",
        ),
        (
            "canonical_default_local",
            "data/delta/ethereum_logs",
            "data/analytics/ethereum_analytics.duckdb",
            "local canonical default name",
        ),
    ]
    for name, delta_relative, duckdb_relative, source in definitions:
        delta_path = first_existing(relative_candidates(delta_relative))
        duckdb_path = first_existing(relative_candidates(duckdb_relative))
        if delta_path and duckdb_path:
            pairs.append(CandidatePair(name, delta_path, duckdb_path, source))

    return pairs


def delta_inventory(pair: CandidatePair) -> dict[str, Any]:
    try:
        table = DeltaTable(str(pair.delta_path))
        schema_records, schema_current = compare_schema_fields(table.schema().to_pyarrow())
        columns = {record["column_name"] for record in schema_records if record["actual_type"]}
        ts_column = "block_timestamp_utc" if "block_timestamp_utc" in columns else "block_timestamp"
        ingested_column = "ingested_at_utc" if "ingested_at_utc" in columns else "ingested_at"
        interval_column = "interval_start_utc" if "interval_start_utc" in columns else None
        con = duckdb.connect()
        con.register("raw_logs", table.to_pyarrow_dataset())
        summary_select = [
            "count(*) as row_count",
            f"min({ts_column}) as first_event_utc",
            f"max({ts_column}) as latest_event_utc",
            f"max({ingested_column}) as latest_ingested_utc",
        ]
        if interval_column:
            summary_select.extend(
                [
                    f"count(distinct {interval_column}) as interval_count",
                    f"min({interval_column}) as first_interval_start_utc",
                    f"max({interval_column}) as latest_interval_start_utc",
                ]
            )
        summary, error = fetch_records(con, f"select {', '.join(summary_select)} from raw_logs")
        if error:
            raise RuntimeError(error)
        row = summary[0]
        return {
            "pair": pair.name,
            "delta_path": str(pair.delta_path),
            "delta_version": table.version(),
            "schema_current": schema_current,
            "row_count": row.get("row_count"),
            "interval_count": row.get("interval_count"),
            "first_event_utc": row.get("first_event_utc"),
            "latest_event_utc": row.get("latest_event_utc"),
            "latest_ingested_utc": row.get("latest_ingested_utc"),
            "first_interval_start_utc": row.get("first_interval_start_utc"),
            "latest_interval_start_utc": row.get("latest_interval_start_utc"),
            "error": None,
        }
    except Exception as exc:
        return {
            "pair": pair.name,
            "delta_path": str(pair.delta_path),
            "delta_version": None,
            "schema_current": False,
            "row_count": None,
            "interval_count": None,
            "first_event_utc": None,
            "latest_event_utc": None,
            "latest_ingested_utc": None,
            "first_interval_start_utc": None,
            "latest_interval_start_utc": None,
            "error": f"{type(exc).__name__}: {exc}",
        }


def duckdb_inventory(pair: CandidatePair) -> dict[str, Any]:
    relation_results: dict[str, Any] = {}
    try:
        con = duckdb.connect(str(pair.duckdb_path), read_only=True)
        relations, relation_error = fetch_records(
            con,
            """
            select table_name, table_type
            from information_schema.tables
            where table_schema = 'main'
            order by table_name
            """,
        )
        relation_names = {row["table_name"] for row in relations}
        for relation in EXPECTED_RELATIONS:
            if relation not in relation_names:
                relation_results[relation] = "MISSING"
                continue
            count_value, count_error = fetch_scalar(con, f"select count(*) from main.{relation}")
            relation_results[relation] = count_value if count_error is None else count_error
        return {
            "pair": pair.name,
            "duckdb_path": str(pair.duckdb_path),
            "relation_count": len(relations),
            "relations_error": relation_error,
            "ethereum_logs": relation_results.get("ethereum_logs"),
            "erc20_transfers": relation_results.get("erc20_transfers"),
            "tether_treasury_flow": relation_results.get("tether_treasury_flow"),
            "quality_summary": relation_results.get("tether_treasury_flow_quality_summary"),
            "error": None,
        }
    except Exception as exc:
        return {
            "pair": pair.name,
            "duckdb_path": str(pair.duckdb_path),
            "relation_count": None,
            "relations_error": None,
            "ethereum_logs": None,
            "erc20_transfers": None,
            "tether_treasury_flow": None,
            "quality_summary": None,
            "error": f"{type(exc).__name__}: {exc}",
        }


PAIRS = candidate_pairs()
if not PAIRS:
    raise FileNotFoundError("검증 가능한 Delta/DuckDB pair를 찾지 못했다.")

DELTA_INVENTORY = [delta_inventory(pair) for pair in PAIRS]
DUCKDB_INVENTORY = [duckdb_inventory(pair) for pair in PAIRS]


def downstream_core_ok(duckdb_row: dict[str, Any]) -> bool:
    erc20_count = duckdb_row.get("erc20_transfers")
    flow_count = duckdb_row.get("tether_treasury_flow")
    return isinstance(erc20_count, int) and erc20_count > 0 and isinstance(flow_count, int) and flow_count > 0


inventory_by_pair = {row["pair"]: row for row in DUCKDB_INVENTORY}
selected_pair = None
if PAIRS[0].name == "env_override":
    selected_pair = PAIRS[0]
else:
    for pair in PAIRS:
        delta_row = next(row for row in DELTA_INVENTORY if row["pair"] == pair.name)
        duckdb_row = inventory_by_pair[pair.name]
        if delta_row["schema_current"] and delta_row["row_count"] and downstream_core_ok(duckdb_row):
            selected_pair = pair
            break
    if selected_pair is None:
        selected_pair = PAIRS[0]

SELECTED_PAIR = selected_pair
SELECTED_DELTA_PATH = SELECTED_PAIR.delta_path
SELECTED_DUCKDB_PATH = SELECTED_PAIR.duckdb_path

for row in DELTA_INVENTORY:
    row["selected"] = row["pair"] == SELECTED_PAIR.name
for row in DUCKDB_INVENTORY:
    row["selected"] = row["pair"] == SELECTED_PAIR.name

df_delta_inventory = display_dataframe("Delta 후보 인벤토리", DELTA_INVENTORY)
df_duckdb_inventory = display_dataframe("DuckDB 후보 인벤토리", DUCKDB_INVENTORY)
display(
    Markdown(
        "### 선택된 검증 pair\n"
        "```text\n"
        f"pair = {SELECTED_PAIR.name}\n"
        f"delta_path = {SELECTED_DELTA_PATH}\n"
        f"duckdb_path = {SELECTED_DUCKDB_PATH}\n"
        "```"
    )
)

### Delta 후보 인벤토리

,pair,delta_path,delta_version,schema_current,row_count,interval_count,first_event_utc,latest_event_utc,latest_ingested_utc,first_interval_start_utc,latest_interval_start_utc,error,selected
0,latest_v2_local,/workspace/data/delta/ethereum_logs_v2,42,True,6956244,43.0,2026-06-20 20:00:11+00:00,2026-06-22 15:59:59+00:00,2026-06-22 16:22:33.761756+00:00,2026-06-20 20:00:00+00:00,2026-06-22 15:00:00+00:00,None,True
1,canonical_default_local,/workspace/data/delta/ethereum_logs,0,False,1,NaN,2024-01-01 00:00:00+00:00,2024-01-01 00:00:00+00:00,2024-01-01 01:00:00+00:00,NaT,NaT,None,False


### DuckDB 후보 인벤토리

,pair,duckdb_path,relation_count,relations_error,ethereum_logs,erc20_transfers,tether_treasury_flow,quality_summary,error,selected
0,latest_v2_local,/workspace/data/analytics/ethereum_analytics_v...,13,None,IOException: IO Error: DeltaKernel InvalidTabl...,6079379,2,1,None,True
1,canonical_default_local,/workspace/data/analytics/ethereum_analytics.d...,6,None,MISSING,1,1,MISSING,None,False


### 선택된 검증 pair
```text
pair = latest_v2_local
delta_path = /workspace/data/delta/ethereum_logs_v2
duckdb_path = /workspace/data/analytics/ethereum_analytics_v2.duckdb
```

## 2. Python source raw 계약과 선택된 Delta schema 비교

In [14]:
DELTA_TABLE = DeltaTable(str(SELECTED_DELTA_PATH))
SCHEMA_CONTRACT_RECORDS, RAW_SCHEMA_CURRENT = compare_schema_fields(DELTA_TABLE.schema().to_pyarrow())
SCHEMA_STATUS = "VERIFIED" if RAW_SCHEMA_CURRENT else "PARTIALLY VERIFIED"

display(
    Markdown(
        "### Raw schema 계약 판정\n"
        "```text\n"
        f"delta_version = {DELTA_TABLE.version()}\n"
        f"schema_status = {SCHEMA_STATUS}\n"
        "```"
    )
)
df_schema_contract = display_dataframe("Schema contract comparison", sorted(SCHEMA_CONTRACT_RECORDS, key=lambda row: (row["status"], row["column_name"])))

### Raw schema 계약 판정
```text
delta_version = 42
schema_status = VERIFIED
```

### Schema contract comparison

,column_name,expected_type,actual_type,expected_nullable,actual_nullable,status
0,block_date_utc,date32[day],date32[day],False,False,PASS
1,block_hash,string,string,False,False,PASS
2,block_number,int64,int64,False,False,PASS
3,block_timestamp_utc,"timestamp[us, tz=UTC]","timestamp[us, tz=UTC]",False,False,PASS
4,chain_id,int64,int64,False,False,PASS
5,contract_address,string,string,False,False,PASS
6,data_raw,string,string,False,False,PASS
7,data_uint256_decimal_text,string,string,True,True,PASS
8,data_uint256_decode_status,string,string,False,False,PASS
9,ingested_at_utc,"timestamp[us, tz=UTC]","timestamp[us, tz=UTC]",False,False,PASS


## 3. Raw Delta 추출, 중복, 시간 범위 확인

In [15]:
RAW_DATASET = DELTA_TABLE.to_pyarrow_dataset()
RAW_CON = duckdb.connect()
RAW_CON.register("raw_logs", RAW_DATASET)
ACTUAL_COLUMNS = {record["column_name"] for record in SCHEMA_CONTRACT_RECORDS if record["actual_type"]}

BLOCK_TIMESTAMP_COLUMN = "block_timestamp_utc" if "block_timestamp_utc" in ACTUAL_COLUMNS else "block_timestamp"
INGESTED_AT_COLUMN = "ingested_at_utc" if "ingested_at_utc" in ACTUAL_COLUMNS else "ingested_at"
CONTRACT_COLUMN = "contract_address" if "contract_address" in ACTUAL_COLUMNS else "address"
DATA_COLUMN = "data_raw" if "data_raw" in ACTUAL_COLUMNS else "data"
INTERVAL_START_COLUMN = "interval_start_utc" if "interval_start_utc" in ACTUAL_COLUMNS else None
INTERVAL_END_COLUMN = "interval_end_utc" if "interval_end_utc" in ACTUAL_COLUMNS else None

RAW_COUNT, raw_count_error = fetch_scalar(RAW_CON, "select count(*) from raw_logs")
RAW_DUPLICATE_KEY_COUNT, duplicate_error = fetch_scalar(
    RAW_CON,
    f"""
    select count(*)
    from (
        select {', '.join(NATURAL_KEY_COLUMNS)}
        from raw_logs
        group by {', '.join(str(index) for index in range(1, len(NATURAL_KEY_COLUMNS) + 1))}
        having count(*) > 1
    )
    """,
)

summary_select = [
    "count(*) as raw_row_count",
    "count(distinct transaction_hash) as transaction_count",
    "min(block_number) as min_block_number",
    "max(block_number) as max_block_number",
    f"min({BLOCK_TIMESTAMP_COLUMN}) as first_block_timestamp_utc",
    f"max({BLOCK_TIMESTAMP_COLUMN}) as latest_block_timestamp_utc",
    f"max({INGESTED_AT_COLUMN}) as latest_ingested_at_utc",
]
if INTERVAL_START_COLUMN:
    summary_select.extend(
        [
            f"count(distinct {INTERVAL_START_COLUMN}) as interval_count",
            f"min({INTERVAL_START_COLUMN}) as first_interval_start_utc",
            f"max({INTERVAL_START_COLUMN}) as latest_interval_start_utc",
        ]
    )
RAW_SUMMARY_RECORDS, raw_summary_error = fetch_records(
    RAW_CON,
    f"select {', '.join(summary_select)} from raw_logs",
)
RAW_SUMMARY = RAW_SUMMARY_RECORDS[0] if RAW_SUMMARY_RECORDS else {}
RAW_LATEST_EVENT_UTC = as_utc_datetime(RAW_SUMMARY.get("latest_block_timestamp_utc"))
RAW_LATEST_INGESTED_UTC = as_utc_datetime(RAW_SUMMARY.get("latest_ingested_at_utc"))

RAW_HEALTH_RECORDS = [
    {
        "check": "raw row count",
        "actual": RAW_COUNT,
        "expected": "> 0",
        "status": "PASS" if isinstance(RAW_COUNT, int) and RAW_COUNT > 0 and raw_count_error is None else "FAIL",
        "detail": raw_count_error,
    },
    {
        "check": "raw natural key duplicate",
        "actual": RAW_DUPLICATE_KEY_COUNT,
        "expected": 0,
        "status": "PASS" if RAW_DUPLICATE_KEY_COUNT == 0 and duplicate_error is None else "FAIL",
        "detail": duplicate_error,
    },
    {
        "check": "raw schema matches current Python source",
        "actual": SCHEMA_STATUS,
        "expected": "VERIFIED",
        "status": "PASS" if RAW_SCHEMA_CURRENT else "PARTIALLY VERIFIED",
        "detail": None,
    },
]

if raw_summary_error:
    display(Markdown(f"### Raw summary query error\n```text\n{raw_summary_error}\n```"))
df_raw_health = display_dataframe("Raw health checks", RAW_HEALTH_RECORDS)
df_raw_summary = display_dataframe("Raw extraction summary", RAW_SUMMARY_RECORDS)

### Raw health checks

,check,actual,expected,status,detail
0,raw row count,6956244,> 0,PASS,None
1,raw natural key duplicate,0,0,PASS,None
2,raw schema matches current Python source,VERIFIED,VERIFIED,PASS,None


### Raw extraction summary

,raw_row_count,transaction_count,min_block_number,max_block_number,first_block_timestamp_utc,latest_block_timestamp_utc,latest_ingested_at_utc,interval_count,first_interval_start_utc,latest_interval_start_utc
0,6956244,1446350,25361048,25374197,2026-06-20 20:00:11+00:00,2026-06-22 15:59:59+00:00,2026-06-22 16:22:33.761756+00:00,43,2026-06-20 20:00:00+00:00,2026-06-22 15:00:00+00:00


## 4. Raw sample과 hourly 시계열성 확인

In [16]:
sample_columns = [
    "chain_id",
    "block_number",
    f"{BLOCK_TIMESTAMP_COLUMN} as block_timestamp_utc",
    "transaction_hash",
    "log_index",
    f"{CONTRACT_COLUMN} as contract_address",
    "topic0",
    "topic1",
    "topic2",
    "topic3",
    f"{DATA_COLUMN} as data_raw",
]
if INTERVAL_START_COLUMN and INTERVAL_END_COLUMN:
    sample_columns.extend(
        [
            f"{INTERVAL_START_COLUMN} as interval_start_utc",
            f"{INTERVAL_END_COLUMN} as interval_end_utc",
        ]
    )
if "data_uint256_decimal_text" in ACTUAL_COLUMNS:
    sample_columns.append("data_uint256_decimal_text")
if "data_uint256_decode_status" in ACTUAL_COLUMNS:
    sample_columns.append("data_uint256_decode_status")

RAW_SAMPLE_RECORDS, raw_sample_error = fetch_records(
    RAW_CON,
    f"""
    select {', '.join(sample_columns)}
    from raw_logs
    order by {BLOCK_TIMESTAMP_COLUMN} desc, log_index desc
    limit 10
    """,
)

if INTERVAL_START_COLUMN:
    RAW_INTERVAL_SERIES_RECORDS, interval_series_error = fetch_records(
        RAW_CON,
        f"""
        select
            {INTERVAL_START_COLUMN} as interval_start_utc,
            count(*) as row_count,
            count(distinct transaction_hash) as transaction_count,
            min({BLOCK_TIMESTAMP_COLUMN}) as first_event_utc,
            max({BLOCK_TIMESTAMP_COLUMN}) as latest_event_utc,
            min(block_number) as min_block_number,
            max(block_number) as max_block_number
        from raw_logs
        group by 1
        order by 1 desc
        limit {SERIES_LOOKBACK_ROWS}
        """,
    )
    RAW_LOAD_TREND_RECORDS, load_trend_error = fetch_records(
        RAW_CON,
        f"""
        with hourly as (
            select
                {INTERVAL_START_COLUMN} as interval_start_utc,
                count(*) as row_count,
                count(distinct transaction_hash) as transaction_count,
                min({BLOCK_TIMESTAMP_COLUMN}) as first_event_utc,
                max({BLOCK_TIMESTAMP_COLUMN}) as latest_event_utc,
                min({INGESTED_AT_COLUMN}) as first_ingested_at_utc,
                max({INGESTED_AT_COLUMN}) as latest_ingested_at_utc
            from raw_logs
            group by 1
        ), trend as (
            select
                interval_start_utc,
                row_count,
                transaction_count,
                sum(row_count) over(order by interval_start_utc rows between unbounded preceding and current row) as cumulative_row_count,
                first_event_utc,
                latest_event_utc,
                first_ingested_at_utc,
                latest_ingested_at_utc
            from hourly
        )
        select *
        from trend
        order by interval_start_utc desc
        limit {SERIES_LOOKBACK_ROWS}
        """,
    )
    RAW_GAP_RECORDS, gap_error = fetch_records(
        RAW_CON,
        f"""
        with intervals as (
            select distinct {INTERVAL_START_COLUMN} as interval_start_utc
            from raw_logs
        ), ordered as (
            select
                interval_start_utc,
                lead(interval_start_utc) over(order by interval_start_utc) as next_interval_start_utc
            from intervals
        )
        select
            interval_start_utc as gap_after_interval_start_utc,
            next_interval_start_utc,
            date_diff('hour', interval_start_utc, next_interval_start_utc) as gap_hours
        from ordered
        where next_interval_start_utc is not null
          and date_diff('hour', interval_start_utc, next_interval_start_utc) > 1
        order by interval_start_utc
        """,
    )
    RAW_INTERVAL_GAP_COUNT = len(RAW_GAP_RECORDS)
    RAW_MAX_GAP_HOURS = max((row["gap_hours"] for row in RAW_GAP_RECORDS), default=0)
else:
    RAW_INTERVAL_SERIES_RECORDS = []
    RAW_LOAD_TREND_RECORDS = []
    RAW_GAP_RECORDS = []
    interval_series_error = "interval_start_utc column missing"
    load_trend_error = "interval_start_utc column missing"
    gap_error = "interval_start_utc column missing"
    RAW_INTERVAL_GAP_COUNT = None
    RAW_MAX_GAP_HOURS = None

RAW_EVENT_HOUR_SERIES_RECORDS, event_hour_error = fetch_records(
    RAW_CON,
    f"""
    select
        date_trunc('hour', {BLOCK_TIMESTAMP_COLUMN}) as event_hour_utc,
        count(*) as row_count,
        count(distinct transaction_hash) as transaction_count
    from raw_logs
    group by 1
    order by 1 desc
    limit {SERIES_LOOKBACK_ROWS}
    """,
)

if raw_sample_error:
    display(Markdown(f"### Raw sample query error\n```text\n{raw_sample_error}\n```"))
df_raw_sample = display_dataframe("Latest raw sample", RAW_SAMPLE_RECORDS, limit=10)
if interval_series_error:
    display(Markdown(f"### Interval series warning\n```text\n{interval_series_error}\n```"))
df_interval_series = display_dataframe("Latest interval_start_utc series", RAW_INTERVAL_SERIES_RECORDS, limit=SERIES_LOOKBACK_ROWS)
if load_trend_error:
    display(Markdown(f"### Load trend warning\n```text\n{load_trend_error}\n```"))
df_load_trend = display_dataframe("시간대별 데이터 적재 추이(DB 적재 사항이 시계열성임을 확인)", RAW_LOAD_TREND_RECORDS, limit=SERIES_LOOKBACK_ROWS)
df_hourly_gaps = display_dataframe("Raw hourly gap list", RAW_GAP_RECORDS)
if event_hour_error:
    display(Markdown(f"### Event-hour series warning\n```text\n{event_hour_error}\n```"))
df_event_hour_series = display_dataframe("Latest event-hour series", RAW_EVENT_HOUR_SERIES_RECORDS, limit=SERIES_LOOKBACK_ROWS)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Latest raw sample

,chain_id,block_number,block_timestamp_utc,transaction_hash,log_index,contract_address,topic0,topic1,topic2,topic3,data_raw,interval_start_utc,interval_end_utc,data_uint256_decimal_text,data_uint256_decode_status
0,1,25374197,2026-06-22 15:59:59+00:00,0xc5377bf8fd7d57725bfb08decea7c432c29adb43718b...,515,0xdac17f958d2ee523a2206206994597c13d831ec7,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x0000000000000000000000002a924f5f2fa0a9906c6a...,0x00000000000000000000000075a5a265d21bc89ec130...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,8243853,DECIMAL38_AVAILABLE
1,1,25374197,2026-06-22 15:59:59+00:00,0xc5377bf8fd7d57725bfb08decea7c432c29adb43718b...,514,0x7e2ac793f3e692f388e66c7dc28f739d13b0b71a,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x00000000000000000000000075a5a265d21bc89ec130...,0x0000000000000000000000002a924f5f2fa0a9906c6a...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,1969246316553576309,DECIMAL38_AVAILABLE
2,1,25374197,2026-06-22 15:59:59+00:00,0x3c2d0674fdb6c678e5445c54f607c77f815af69431ca...,513,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x000000000000000000000000e8736af1926e9d8f5a66...,0x00000000000000000000000086a067030a9668c13ff2...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,70929037833,DECIMAL38_AVAILABLE
3,1,25374197,2026-06-22 15:59:59+00:00,0xfe799c4ebcfa6694766837e2739e58a93d099054ebaf...,512,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x0000000000000000000000000bd0c9b6f93c2b22bffc...,0x00000000000000000000000031f8d5b83802b0a33d36...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,1000000,DECIMAL38_AVAILABLE
4,1,25374197,2026-06-22 15:59:59+00:00,0x8e1dc468db89673c197d29bf31a2fa6c1c67b82ff3d8...,511,0xdac17f958d2ee523a2206206994597c13d831ec7,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x0000000000000000000000002ee9f7fe2f64d4a1aac6...,0x000000000000000000000000ac21e52fd16fdcecff95...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,58550000,DECIMAL38_AVAILABLE
5,1,25374197,2026-06-22 15:59:59+00:00,0xa1ab2d93fc4929f189a371a3d66eaad77a2f0a4348bc...,510,0x6c3ea9036406852006290770bedfcaba0e23a0e8,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x000000000000000000000000587e5b7e4818643cf940...,0x0000000000000000000000001d69cc3c43fb615c04c6...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,1000000,DECIMAL38_AVAILABLE
6,1,25374197,2026-06-22 15:59:59+00:00,0xe9f761a6cb28a181bb5f9cdfbbd1fa22cce274852cce...,505,0xdac17f958d2ee523a2206206994597c13d831ec7,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x0000000000000000000000009f4c982fc3af26cd1c09...,0x0000000000000000000000007b65887e6f79f5d9a053...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,68000000,DECIMAL38_AVAILABLE
7,1,25374197,2026-06-22 15:59:59+00:00,0xc24ed4b1f966eb5a1d06a2ac93538c7e560223d1e29b...,501,0x07041776f5007aca2a54844f50503a18a72a8b68,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x00000000000000000000000009cd15c9a337c85df9e5...,0x0000000000000000000000008b1f6cb5d062aa2ce8d5...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,478008,DECIMAL38_AVAILABLE
8,1,25374197,2026-06-22 15:59:59+00:00,0xc24ed4b1f966eb5a1d06a2ac93538c7e560223d1e29b...,497,0x07041776f5007aca2a54844f50503a18a72a8b68,0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4...,0x00000000000000000000000009cd15c9a337c85df9e5...,0x000000000000000000000000182a181f2e6b7087b9a4...,None,0x00000000000000000000000000000000000000000000...,2026-06-22 15:00:00+00:00,2026-06-22 16:00:00+00:00,8889214,D

### Latest interval_start_utc series

,interval_start_utc,row_count,transaction_count,first_event_utc,latest_event_utc,min_block_number,max_block_number
0,2026-06-22 15:00:00+00:00,107307,50879,2026-06-22 15:00:11+00:00,2026-06-22 15:59:59+00:00,25373898,25374197
1,2026-06-22 14:00:00+00:00,141932,54867,2026-06-22 14:00:11+00:00,2026-06-22 14:59:59+00:00,25373599,25373897
2,2026-06-22 13:00:00+00:00,155609,54317,2026-06-22 13:00:11+00:00,2026-06-22 13:59:59+00:00,25373301,25373598
3,2026-06-22 11:00:00+00:00,143274,46170,2026-06-22 11:00:11+00:00,2026-06-22 11:59:59+00:00,25372701,25373000
4,2026-06-22 10:00:00+00:00,174465,45868,2026-06-22 10:00:11+00:00,2026-06-22 10:59:59+00:00,25372402,25372700
5,2026-06-22 09:00:00+00:00,150725,46023,2026-06-22 09:00:11+00:00,2026-06-22 09:59:59+00:00,25372103,25372401
6,2026-06-22 08:00:00+00:00,169451,42942,2026-06-22 08:00:11+00:00,2026-06-22 08:59:59+00:00,25371803,25372102
7,2026-06-22 07:00:00+00:00,156301,38210,2026-06-22 07:00:11+00:00,2026-06-22 07:59:59+00:00,25371503,25371802
8,2026-06-22 06:00:00+00:00,166570,36887,2026-06-22 06:00:11+00:00,2026-06-22 06:59:59+00:00,25371204,25371502
9,2026-06-22 05:00:00+00:00,181242,32987,2026-06-22 05:00:11+00:00,2026-06-22 05:59:59+00:00,25370904,25371203


...

,interval_start_utc,row_count,transaction_count,cumulative_row_count,first_event_utc,latest_event_utc,first_ingested_at_utc,latest_ingested_at_utc
0,2026-06-22 15:00:00+00:00,107307,50879,6956244,2026-06-22 15:00:11+00:00,2026-06-22 15:59:59+00:00,2026-06-22 16:22:33.761756+00:00,2026-06-22 16:22:33.761756+00:00
1,2026-06-22 14:00:00+00:00,141932,54867,6848937,2026-06-22 14:00:11+00:00,2026-06-22 14:59:59+00:00,2026-06-22 15:25:05.500466+00:00,2026-06-22 15:25:05.500466+00:00
2,2026-06-22 13:00:00+00:00,155609,54317,6707005,2026-06-22 13:00:11+00:00,2026-06-22 13:59:59+00:00,2026-06-22 14:43:07.237528+00:00,2026-06-22 14:43:07.237528+00:00
3,2026-06-22 11:00:00+00:00,143274,46170,6551396,2026-06-22 11:00:11+00:00,2026-06-22 11:59:59+00:00,2026-06-22 14:40:43.619419+00:00,2026-06-22 14:40:43.619419+00:00
4,2026-06-22 10:00:00+00:00,174465,45868,6408122,2026-06-22 10:00:11+00:00,2026-06-22 10:59:59+00:00,2026-06-22 11:21:25.218302+00:00,2026-06-22 11:21:25.218302+00:00
5,2026-06-22 09:00:00+00:00,150725,46023,6233657,2026-06-22 09:00:11+00:00,2026-06-22 09:59:59+00:00,2026-06-22 10:22:17.000637+00:00,2026-06-22 10:22:17.000637+00:00
6,2026-06-22 08:00:00+00:00,169451,42942,6082932,2026-06-22 08:00:11+00:00,2026-06-22 08:59:59+00:00,2026-06-22 09:24:17.983959+00:00,2026-06-22 09:24:17.983959+00:00
7,2026-06-22 07:00:00+00:00,156301,38210,5913481,2026-06-22 07:00:11+00:00,2026-06-22 07:59:59+00:00,2026-06-22 08:23:55.343412+00:00,2026-06-22 08:23:55.343412+00:00
8,2026-06-22 06:00:00+00:00,166570,36887,5757180,2026-06-22 06:00:11+00:00,2026-06-22 06:59:59+00:00,2026-06-22 07:28:47.656627+00:00,2026-06-22 07:28:47.656627+00:00
9,2026-06-22 05:00:00+00:00,181242,32987,5590610,2026-06-22 05:00:11+00:00,2026-06-22 05:59:59+00:00,2026-06-22 06:23:48.147085+00:00,2026-06-22 06:23:48.147085+00:00


### Raw hourly gap list

,gap_after_interval_start_utc,next_interval_start_utc,gap_hours
0,2026-06-22 11:00:00+00:00,2026-06-22 13:00:00+00:00,2


### Latest event-hour series

,event_hour_utc,row_count,transaction_count
0,2026-06-22 15:00:00+00:00,107307,50879
1,2026-06-22 14:00:00+00:00,141932,54867
2,2026-06-22 13:00:00+00:00,155609,54317
3,2026-06-22 11:00:00+00:00,143274,46170
4,2026-06-22 10:00:00+00:00,174465,45868
5,2026-06-22 09:00:00+00:00,150725,46023
6,2026-06-22 08:00:00+00:00,169451,42942
7,2026-06-22 07:00:00+00:00,156301,38210
8,2026-06-22 06:00:00+00:00,166570,36887
9,2026-06-22 05:00:00+00:00,181242,32987


## 5. DuckDB relation 추출 상태 확인

In [17]:
ANALYTICS_CON = duckdb.connect(str(SELECTED_DUCKDB_PATH), read_only=True)
RELATION_RECORDS, relations_error = fetch_records(
    ANALYTICS_CON,
    """
    select table_schema, table_name, table_type
    from information_schema.tables
    where table_schema = 'main'
    order by table_name
    """,
)
RELATION_NAMES = {row["table_name"] for row in RELATION_RECORDS}

RELATION_STATUS_RECORDS: list[dict[str, Any]] = []
for relation in EXPECTED_RELATIONS:
    exists = relation in RELATION_NAMES
    row_count = None
    query_error = None
    if exists:
        row_count, query_error = fetch_scalar(ANALYTICS_CON, f"select count(*) from main.{relation}")
    RELATION_STATUS_RECORDS.append(
        {
            "relation": relation,
            "expected_role": "staging" if relation == "ethereum_logs" else "downstream",
            "exists": exists,
            "row_count": row_count,
            "query_error": query_error,
            "status": "PASS" if exists and query_error is None else "PARTIALLY VERIFIED",
        }
    )

DB_DOWNSTREAM_CORE_OK = all(
    row["exists"] and row["query_error"] is None and isinstance(row["row_count"], int) and row["row_count"] > 0
    for row in RELATION_STATUS_RECORDS
    if row["relation"] in {"erc20_transfers", "tether_treasury_flow"}
)
DB_STAGING_QUERY_ERROR = next(
    (row["query_error"] for row in RELATION_STATUS_RECORDS if row["relation"] == "ethereum_logs"),
    None,
)

if relations_error:
    display(Markdown(f"### Relation inventory error\n```text\n{relations_error}\n```"))
df_duckdb_relations = display_dataframe("DuckDB relations", RELATION_RECORDS)
df_duckdb_extraction = display_dataframe("DuckDB extraction status", RELATION_STATUS_RECORDS)

### DuckDB relations

,table_schema,table_name,table_type
0,main,erc20_amount_numeric_status_integrity,VIEW
1,main,erc20_transfer_integrity,VIEW
2,main,erc20_transfers,BASE TABLE
3,main,ethereum_logs,VIEW
4,main,ethereum_logs_uint256_contract,VIEW
5,main,non_usdt_amount_usdt_null,VIEW
6,main,stg_ethereum_logs,VIEW
7,main,tether_treasury_flow,BASE TABLE
8,main,tether_treasury_flow_quality_summary,VIEW
9,main,treasury_flow_integrity,VIEW


### DuckDB extraction status

,relation,expected_role,exists,row_count,query_error,status
0,ethereum_logs,staging,True,NaN,IOException: IO Error: DeltaKernel InvalidTabl...,PARTIALLY VERIFIED
1,erc20_transfers,downstream,True,6079379.0,NaN,PASS
2,tether_treasury_flow,downstream,True,2.0,NaN,PASS
3,tether_treasury_flow_quality_summary,downstream,True,1.0,NaN,PASS


## 6. Silver/Gold schema와 추출 sample

In [18]:
def relation_columns(relation: str) -> tuple[list[str], str | None]:
    records, error = fetch_records(ANALYTICS_CON, f"describe main.{relation}")
    if error:
        return [], error
    return [row["column_name"] for row in records], None


def select_existing_columns(available: list[str], desired: list[str]) -> list[str]:
    return [column for column in desired if column in available]


DOWNSTREAM_SCHEMA_RECORDS: list[dict[str, Any]] = []
DOWNSTREAM_SAMPLE_ERRORS: list[dict[str, Any]] = []

for relation in ["erc20_transfers", "tether_treasury_flow", "tether_treasury_flow_quality_summary"]:
    if relation not in RELATION_NAMES:
        DOWNSTREAM_SAMPLE_ERRORS.append({"relation": relation, "error": "relation missing"})
        continue
    schema_records, schema_error = fetch_records(ANALYTICS_CON, f"describe main.{relation}")
    if schema_error:
        DOWNSTREAM_SAMPLE_ERRORS.append({"relation": relation, "error": schema_error})
    for record in schema_records:
        DOWNSTREAM_SCHEMA_RECORDS.append({"relation": relation, **record})

ERC20_COLUMNS, erc20_columns_error = relation_columns("erc20_transfers") if "erc20_transfers" in RELATION_NAMES else ([], "relation missing")
erc20_desired = [
    "chain_id",
    "block_number",
    "block_timestamp_utc",
    "interval_start_utc",
    "interval_end_utc",
    "transaction_hash",
    "log_index",
    "contract_address",
    "from_address",
    "to_address",
    "amount_numeric_status",
    "amount_usdt",
    "ingested_at_utc",
]
erc20_selected = select_existing_columns(ERC20_COLUMNS, erc20_desired)
if erc20_selected:
    order_column = "block_timestamp_utc" if "block_timestamp_utc" in ERC20_COLUMNS else erc20_selected[0]
    ERC20_SAMPLE_RECORDS, erc20_sample_error = fetch_records(
        ANALYTICS_CON,
        f"select {', '.join(erc20_selected)} from main.erc20_transfers order by {order_column} desc limit 10",
    )
else:
    ERC20_SAMPLE_RECORDS = []
    erc20_sample_error = erc20_columns_error or "erc20_transfers has no selectable columns"

FLOW_COLUMNS, flow_columns_error = relation_columns("tether_treasury_flow") if "tether_treasury_flow" in RELATION_NAMES else ([], "relation missing")
flow_desired = [
    "chain_id",
    "contract_address",
    "treasury_address",
    "hour_start_utc",
    "direction",
    "transfer_count",
    "total_amount_raw",
    "total_amount_usdt",
    "source_interval_start_utc",
    "updated_at_utc",
]
flow_selected = select_existing_columns(FLOW_COLUMNS, flow_desired)
if flow_selected:
    order_column = "hour_start_utc" if "hour_start_utc" in FLOW_COLUMNS else flow_selected[0]
    FLOW_SAMPLE_RECORDS, flow_sample_error = fetch_records(
        ANALYTICS_CON,
        f"select {', '.join(flow_selected)} from main.tether_treasury_flow order by {order_column} desc limit 10",
    )
else:
    FLOW_SAMPLE_RECORDS = []
    flow_sample_error = flow_columns_error or "tether_treasury_flow has no selectable columns"

QUALITY_COLUMNS, quality_columns_error = relation_columns("tether_treasury_flow_quality_summary") if "tether_treasury_flow_quality_summary" in RELATION_NAMES else ([], "relation missing")
if QUALITY_COLUMNS:
    QUALITY_SAMPLE_RECORDS, quality_sample_error = fetch_records(
        ANALYTICS_CON,
        "select * from main.tether_treasury_flow_quality_summary limit 5",
    )
else:
    QUALITY_SAMPLE_RECORDS = []
    quality_sample_error = quality_columns_error

if DOWNSTREAM_SAMPLE_ERRORS:
    df_downstream_schema_errors = display_dataframe("Downstream schema errors", DOWNSTREAM_SAMPLE_ERRORS)
df_downstream_schema = display_dataframe("Downstream schema", DOWNSTREAM_SCHEMA_RECORDS, limit=80)
if erc20_sample_error:
    display(Markdown(f"### erc20_transfers sample warning\n```text\n{erc20_sample_error}\n```"))
df_erc20_sample = display_dataframe("erc20_transfers latest sample", ERC20_SAMPLE_RECORDS, limit=10)
if flow_sample_error:
    display(Markdown(f"### tether_treasury_flow sample warning\n```text\n{flow_sample_error}\n```"))
df_flow_sample = display_dataframe("tether_treasury_flow sample", FLOW_SAMPLE_RECORDS, limit=10)
if quality_sample_error:
    display(Markdown(f"### quality summary sample warning\n```text\n{quality_sample_error}\n```"))
df_quality_sample = display_dataframe("tether_treasury_flow_quality_summary sample", QUALITY_SAMPLE_RECORDS, limit=5)

### Downstream schema

,relation,column_name,column_type,null,key,default,extra
0,erc20_transfers,chain_id,BIGINT,YES,None,None,None
1,erc20_transfers,block_number,BIGINT,YES,None,None,None
2,erc20_transfers,block_timestamp_utc,TIMESTAMP,YES,None,None,None
3,erc20_transfers,block_date_utc,DATE,YES,None,None,None
4,erc20_transfers,interval_start_utc,TIMESTAMP,YES,None,None,None
5,erc20_transfers,interval_end_utc,TIMESTAMP,YES,None,None,None
6,erc20_transfers,transaction_hash,VARCHAR,YES,None,None,None
7,erc20_transfers,log_index,BIGINT,YES,None,None,None
8,erc20_transfers,contract_address,VARCHAR,YES,None,None,None
9,erc20_transfers,from_address,VARCHAR,YES,None,None,None


### erc20_transfers latest sample

,chain_id,block_number,block_timestamp_utc,interval_start_utc,interval_end_utc,transaction_hash,log_index,contract_address,from_address,to_address,amount_numeric_status,amount_usdt,ingested_at_utc
0,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0x2014a4775037bbfba27802a18049e0035bfdfdfab645...,72,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0x555ce236c0220695b68341bc48c68d52210cc35b,0x663dc15d3c1ac63ff12e45ab68fea3f0a883c251,DECIMAL38_AVAILABLE,None,2026-06-22 15:25:05.500466
1,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0xbad3021c05e79fc464580dd5a10f0a7f9920c4066133...,181,0xdac17f958d2ee523a2206206994597c13d831ec7,0xe558a259075e28f22feb57561783861ea8c2e4e6,0xbde37e752b36e793a42b503ef9cd3b8741d9c5ed,DECIMAL38_AVAILABLE,29999.000000,2026-06-22 15:25:05.500466
2,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0xe74623ec7939ebf46547b0bfb57a933dc8e290be52a7...,214,0xdac17f958d2ee523a2206206994597c13d831ec7,0xa4021405ca1827427e04b841df20e85454684aee,0x99a94951edaeeb6dd340f88fa97621e90d2803cc,DECIMAL38_AVAILABLE,900.000000,2026-06-22 15:25:05.500466
3,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0xf58f549583ea69baad5018d1b3921e7d311e3221a665...,155,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0xac34fb388f8184e5eafe5d50a5e96086f10efa0c,0xf1e35050adfa2b27d141178d25c9ba206c5e0982,DECIMAL38_AVAILABLE,None,2026-06-22 15:25:05.500466
4,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0xa252e6d2e3933766de93d71c73a55e2a7d24b305ea5e...,169,0xdac17f958d2ee523a2206206994597c13d831ec7,0x3d3beef6f249cc01e2a4f046348634fa8a21c876,0xe45a413f8a0b70215dc90bd7f0680678bbbbaca6,DECIMAL38_AVAILABLE,53.000000,2026-06-22 15:25:05.500466
5,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0xa0cd2e1c77b5fbf9d1a6836294fdc88acf7a3d3c7d2c...,29,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0xa9d1e08c7793af67e9d92fe308d5697fb81d3e43,0x29582d59ed900fa83def5a09be74b9b771a5f2c7,DECIMAL38_AVAILABLE,None,2026-06-22 15:25:05.500466
6,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0x632f247f0b920e40b61a24dac97fbca56dbd9d56727c...,57,0xdac17f958d2ee523a2206206994597c13d831ec7,0x444b92e107c2867a9f7dd4e0c856e5563c901643,0x18e296053cbdf986196903e889b7dca7a73882f6,DECIMAL38_AVAILABLE,133.887681,2026-06-22 15:25:05.500466
7,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0x4b8a1df6ef631ad9f430a297d6bf13a3e2d8b459d3fb...,63,0x6982508145454ce325ddbe47a25d4ec3d2311933,0x6ac01429cfe0a092b2ae34abd79a0a4e9d6e1e1d,0xefef30bd1cca520619306c95091ab18473febc5c,DECIMAL38_AVAILABLE,None,2026-06-22 15:25:05.500466
8,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0x6403dc990db3d68294ed872537d494645bcb2bb82fe8...,142,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,0x4ced68db4c87166a5ec6d8c1a6f57944ef4473d5,0xfdcdc3450643670ee5727e96f5b5e0864663252e,DECIMAL38_AVAILABLE,None,2026-06-22 15:25:05.500466
9,1,25373897,2026-06-22 14:59:59,2026-06-22 14:00:00,2026-06-22 15:00:00,0xbad3021c05e79fc464580dd5a10f0a7f9920c4066133...,182,0xdac17f958d2ee523a2206206994597c13d831ec7,0xe558a259075e28f22feb57561783861ea8c2e4e6,0x4b742ad5ca91969e82aefb80072ae59121a3d72a,DECIMAL38_AVAILABLE,0.772339,2026-06-22 15:25:05.500466


### tether_treasury_flow sample

,chain_id,contract_address,treasury_address,hour_start_utc,direction,transfer_count,total_amount_raw,total_amount_usdt,source_interval_start_utc,updated_at_utc
0,1,0xdac17f958d2ee523a2206206994597c13d831ec7,0x5754284f345afc66a98fbb0a0afe71e0f007b949,2026-06-22 03:00:00,INFLOW,1,250000000000,250000.000000,2026-06-22 03:00:00,2026-06-22 04:25:56.081844+00:00
1,1,0xdac17f958d2ee523a2206206994597c13d831ec7,0x5754284f345afc66a98fbb0a0afe71e0f007b949,2026-06-21 06:00:00,OUTFLOW,1,220000000000000,220000000.000000,2026-06-21 06:00:00,2026-06-21 07:23:56.033151+00:00


### tether_treasury_flow_quality_summary sample

,window_start_utc,window_end_utc,flow_row_count,transfer_count,direction_count,first_hour_start_utc,last_hour_start_utc,total_inflow_raw,total_outflow_raw,total_inflow_usdt,total_outflow_usdt
0,2026-06-22 14:00:00,2026-06-22 15:00:00,0,0,0,None,None,0,0,0.000000,0.000000


## 7. Freshness와 raw → silver 최신성 비교

In [19]:
SILVER_LATEST_EVENT_UTC = None
silver_error = None
if "erc20_transfers" in RELATION_NAMES:
    erc20_ts_column = "block_timestamp_utc" if "block_timestamp_utc" in ERC20_COLUMNS else "block_timestamp" if "block_timestamp" in ERC20_COLUMNS else None
    if erc20_ts_column:
        silver_latest_raw, silver_error = fetch_scalar(ANALYTICS_CON, f"select max({erc20_ts_column}) from main.erc20_transfers")
        SILVER_LATEST_EVENT_UTC = as_utc_datetime(silver_latest_raw)
    else:
        silver_error = "erc20_transfers has no block timestamp column"
else:
    silver_error = "erc20_transfers relation missing"

NOW_UTC = datetime.now(UTC)
RAW_EVENT_AGE_HOURS = hours_between(NOW_UTC, RAW_LATEST_EVENT_UTC)
RAW_INGEST_AGE_HOURS = hours_between(NOW_UTC, RAW_LATEST_INGESTED_UTC)
RAW_TO_SILVER_LAG_HOURS = hours_between(RAW_LATEST_EVENT_UTC, SILVER_LATEST_EVENT_UTC)

freshness_status = "PASS"
if RAW_EVENT_AGE_HOURS is None or RAW_EVENT_AGE_HOURS > FRESHNESS_WARNING_HOURS:
    freshness_status = "PARTIALLY VERIFIED"
if silver_error or SILVER_LATEST_EVENT_UTC is None:
    raw_to_silver_status = "PARTIALLY VERIFIED"
elif SILVER_LATEST_EVENT_UTC >= RAW_LATEST_EVENT_UTC:
    raw_to_silver_status = "PASS"
else:
    raw_to_silver_status = "PARTIALLY VERIFIED"

FRESHNESS_RECORDS = [
    {
        "check": "raw latest event timestamp",
        "utc": RAW_LATEST_EVENT_UTC,
        "age_hours_from_now": RAW_EVENT_AGE_HOURS,
        "status": freshness_status,
        "detail": f"warning threshold={FRESHNESS_WARNING_HOURS}h",
    },
    {
        "check": "raw latest ingested_at timestamp",
        "utc": RAW_LATEST_INGESTED_UTC,
        "age_hours_from_now": RAW_INGEST_AGE_HOURS,
        "status": freshness_status,
        "detail": f"warning threshold={FRESHNESS_WARNING_HOURS}h",
    },
    {
        "check": "silver latest event timestamp",
        "utc": SILVER_LATEST_EVENT_UTC,
        "age_hours_from_now": hours_between(NOW_UTC, SILVER_LATEST_EVENT_UTC),
        "status": "INFO" if silver_error is None else "PARTIALLY VERIFIED",
        "detail": silver_error,
    },
    {
        "check": "raw to silver latest timestamp",
        "utc": None,
        "age_hours_from_now": RAW_TO_SILVER_LAG_HOURS,
        "status": raw_to_silver_status,
        "detail": "lag_hours = raw_latest - silver_latest",
    },
    {
        "check": "live realtime external call",
        "utc": NOW_UTC,
        "age_hours_from_now": 0,
        "status": "NOT VERIFIED",
        "detail": "노트북은 외부 RPC 또는 Airflow scheduler를 새로 호출하지 않고 local freshness만 확인한다.",
    },
]

df_freshness = display_dataframe("Freshness checks", FRESHNESS_RECORDS)

### Freshness checks

,check,utc,age_hours_from_now,status,detail
0,raw latest event timestamp,2026-06-22 15:59:59+00:00,1.183,PASS,warning threshold=24h
1,raw latest ingested_at timestamp,2026-06-22 16:22:33.761756+00:00,0.807,PASS,warning threshold=24h
2,silver latest event timestamp,2026-06-22 14:59:59+00:00,2.183,INFO,NaN
3,raw to silver latest timestamp,NaT,1.000,PARTIALLY VERIFIED,lag_hours = raw_latest - silver_latest
4,live realtime external call,2026-06-22 17:10:57.536119+00:00,0.000,NOT VERIFIED,노트북은 외부 RPC 또는 Airflow scheduler를 새로 호출하지 않고 l...


## 8. 최종 판정

In [20]:
DOWNSTREAM_ROW_COUNTS = {row["relation"]: row["row_count"] for row in RELATION_STATUS_RECORDS}

ACCUMULATION_SUMMARY_RECORDS = [
    {
        "metric": "selected raw Delta pair",
        "value": SELECTED_PAIR.name,
        "status": "INFO",
        "detail": str(SELECTED_DELTA_PATH),
    },
    {
        "metric": "raw accumulated rows",
        "value": RAW_COUNT,
        "status": "PASS" if isinstance(RAW_COUNT, int) and RAW_COUNT > 0 else "FAIL",
        "detail": "Delta row count from selected local raw table",
    },
    {
        "metric": "raw duplicate natural keys",
        "value": RAW_DUPLICATE_KEY_COUNT,
        "status": "PASS" if RAW_DUPLICATE_KEY_COUNT == 0 else "FAIL",
        "detail": f"natural_key={NATURAL_KEY_COLUMNS}",
    },
    {
        "metric": "hourly interval count",
        "value": RAW_SUMMARY.get("interval_count"),
        "status": "PASS" if RAW_INTERVAL_GAP_COUNT == 0 else "PARTIALLY VERIFIED",
        "detail": f"gap_count={RAW_INTERVAL_GAP_COUNT}, max_gap_hours={RAW_MAX_GAP_HOURS}",
    },
    {
        "metric": "latest raw event UTC",
        "value": RAW_LATEST_EVENT_UTC,
        "status": freshness_status,
        "detail": f"age_hours={RAW_EVENT_AGE_HOURS}",
    },
    {
        "metric": "latest raw ingested UTC",
        "value": RAW_LATEST_INGESTED_UTC,
        "status": freshness_status,
        "detail": f"age_hours={RAW_INGEST_AGE_HOURS}",
    },
    {
        "metric": "erc20_transfers rows",
        "value": DOWNSTREAM_ROW_COUNTS.get("erc20_transfers"),
        "status": "PASS" if isinstance(DOWNSTREAM_ROW_COUNTS.get("erc20_transfers"), int) and DOWNSTREAM_ROW_COUNTS["erc20_transfers"] > 0 else "PARTIALLY VERIFIED",
        "detail": str(SELECTED_DUCKDB_PATH),
    },
    {
        "metric": "tether_treasury_flow rows",
        "value": DOWNSTREAM_ROW_COUNTS.get("tether_treasury_flow"),
        "status": "PASS" if isinstance(DOWNSTREAM_ROW_COUNTS.get("tether_treasury_flow"), int) and DOWNSTREAM_ROW_COUNTS["tether_treasury_flow"] > 0 else "PARTIALLY VERIFIED",
        "detail": str(SELECTED_DUCKDB_PATH),
    },
]

df_accumulation_summary = display_dataframe("주요 데이터 누적 현황", ACCUMULATION_SUMMARY_RECORDS, limit=20)


### 주요 데이터 누적 현황

,metric,value,status,detail
0,selected raw Delta pair,latest_v2_local,INFO,/workspace/data/delta/ethereum_logs_v2
1,raw accumulated rows,6956244,PASS,Delta row count from selected local raw table
2,raw duplicate natural keys,0,PASS,"natural_key=('chain_id', 'transaction_hash', '..."
3,hourly interval count,43,PARTIALLY VERIFIED,"gap_count=1, max_gap_hours=2"
4,latest raw event UTC,2026-06-22 15:59:59+00:00,PASS,age_hours=1.183
5,latest raw ingested UTC,2026-06-22 16:22:33.761756+00:00,PASS,age_hours=0.807
6,erc20_transfers rows,6079379,PASS,/workspace/data/analytics/ethereum_analytics_v...
7,tether_treasury_flow rows,2,PASS,/workspace/data/analytics/ethereum_analytics_v...


In [21]:
hard_failures: list[str] = []
partial_reasons: list[str] = []

if not isinstance(RAW_COUNT, int) or RAW_COUNT <= 0:
    hard_failures.append("selected raw Delta row count is zero or unavailable")
if RAW_DUPLICATE_KEY_COUNT != 0:
    hard_failures.append("selected raw Delta natural key duplicates exist")
if not DB_DOWNSTREAM_CORE_OK:
    hard_failures.append("required downstream materialized relations are missing or empty")

if SELECTED_PAIR.name != "canonical_default_local":
    partial_reasons.append(f"selected pair is {SELECTED_PAIR.name}; canonical default path is not the freshest local evidence")
if not RAW_SCHEMA_CURRENT:
    partial_reasons.append("selected raw Delta schema does not match current Python delta_writer contract")
if DB_STAGING_QUERY_ERROR:
    partial_reasons.append("DuckDB staging view ethereum_logs is not queryable in this runtime, likely due to stored absolute Delta path")
if RAW_INTERVAL_GAP_COUNT is None:
    partial_reasons.append("raw interval_start_utc is unavailable, so hourly interval continuity cannot be checked")
elif RAW_INTERVAL_GAP_COUNT > 0:
    partial_reasons.append(f"raw hourly interval series has {RAW_INTERVAL_GAP_COUNT} gap(s), max_gap_hours={RAW_MAX_GAP_HOURS}")
if freshness_status != "PASS":
    partial_reasons.append("latest local raw timestamp is older than the freshness warning threshold")
if raw_to_silver_status != "PASS":
    partial_reasons.append("raw to silver latest timestamp alignment is not fully verified")
partial_reasons.append("live realtime behavior is not verified because this notebook does not call external RPC or Airflow scheduler")

if hard_failures:
    FINAL_STATUS = "NOT VERIFIED"
elif partial_reasons:
    FINAL_STATUS = "PARTIALLY VERIFIED"
else:
    FINAL_STATUS = "VERIFIED"

FINAL_STATUS_RECORDS = [
    {
        "area": "selected dataset pair",
        "status": "VERIFIED" if SELECTED_PAIR.name and not hard_failures else "PARTIALLY VERIFIED",
        "evidence": f"pair={SELECTED_PAIR.name}, delta={SELECTED_DELTA_PATH}, duckdb={SELECTED_DUCKDB_PATH}",
    },
    {
        "area": "raw Delta extraction",
        "status": "VERIFIED" if RAW_SCHEMA_CURRENT and RAW_DUPLICATE_KEY_COUNT == 0 and RAW_COUNT > 0 else "PARTIALLY VERIFIED",
        "evidence": f"rows={RAW_COUNT}, duplicate_keys={RAW_DUPLICATE_KEY_COUNT}, schema_current={RAW_SCHEMA_CURRENT}",
    },
    {
        "area": "DuckDB downstream extraction",
        "status": "VERIFIED" if DB_DOWNSTREAM_CORE_OK and not DB_STAGING_QUERY_ERROR else "PARTIALLY VERIFIED",
        "evidence": RELATION_STATUS_RECORDS,
    },
    {
        "area": "freshness",
        "status": "VERIFIED" if freshness_status == "PASS" and raw_to_silver_status == "PASS" else "PARTIALLY VERIFIED",
        "evidence": FRESHNESS_RECORDS,
    },
    {
        "area": "hourly time series continuity",
        "status": "VERIFIED" if RAW_INTERVAL_GAP_COUNT == 0 else "PARTIALLY VERIFIED",
        "evidence": RAW_GAP_RECORDS,
    },
]

display(
    Markdown(
        "### Notebook validation result\n"
        "```text\n"
        f"final_status = {FINAL_STATUS}\n"
        f"hard_failures = {hard_failures}\n"
        f"partial_reasons = {partial_reasons}\n"
        "```"
    )
)
df_final_status = display_dataframe("Final status by area", FINAL_STATUS_RECORDS)

### Notebook validation result
```text
final_status = PARTIALLY VERIFIED
hard_failures = []
partial_reasons = ['selected pair is latest_v2_local; canonical default path is not the freshest local evidence', 'DuckDB staging view ethereum_logs is not queryable in this runtime, likely due to stored absolute Delta path', 'raw hourly interval series has 1 gap(s), max_gap_hours=2', 'raw to silver latest timestamp alignment is not fully verified', 'live realtime behavior is not verified because this notebook does not call external RPC or Airflow scheduler']
```

### Final status by area

,area,status,evidence
0,selected dataset pair,VERIFIED,"pair=latest_v2_local, delta=/workspace/data/de..."
1,raw Delta extraction,VERIFIED,"rows=6956244, duplicate_keys=0, schema_current..."
2,DuckDB downstream extraction,PARTIALLY VERIFIED,"[{'relation': 'ethereum_logs', 'expected_role'..."
3,freshness,PARTIALLY VERIFIED,"[{'check': 'raw latest event timestamp', 'utc'..."
4,hourly time series continuity,PARTIALLY VERIFIED,[{'gap_after_interval_start_utc': 2026-06-22 1...
